# Reinforcement Learning Trading Agent
**Author:** Felipe Taha Sant'Ana  
**Approach:** Tabular Q-learning agent in a custom Gym-style trading environment

---
## Overview
This project implements a complete RL trading pipeline. The agent observes a discretized state of market indicators (RSI, momentum) and a position flag, then learns through trial-and-error to take BUY / SELL / HOLD actions that maximize portfolio value. The agent is evaluated against three classical baselines: Buy & Hold, MA Crossover, and Random.

## Contents
1. [Market Data Generation](#1)
2. [Trading Environment (Gym-style)](#2)
3. [Q-Learning Agent](#3)
4. [Training](#4)
5. [Backtest & Comparison](#5)
6. [Agent Behavior Analysis](#6)
7. [Conclusions](#7)

---
<a id="1"></a>
## 1. Market Data Generation

We synthesize ~10 years of daily price data with three regimes (bull, bear, sideways) that switch periodically. Technical indicators (SMA, RSI, momentum, volatility) are computed as features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.stats import norm
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary':'#0EA5E9','secondary':'#8B5CF6','accent':'#F59E0B','danger':'#EF4444',
          'success':'#10B981','dark':'#0F172A','teal':'#14B8A6','rose':'#F43F5E'}
palette = list(COLORS.values())
np.random.seed(123)
print("Libraries loaded")

In [ ]:
# ── Generate synthetic price data with regime switching ──────────────────────
n_days = 2500
dates = pd.bdate_range('2015-01-01', periods=n_days)
returns, regime = [], []
current_regime, days_in_regime = 'bull', 0
regime_duration = np.random.randint(80, 250)

for i in range(n_days):
    if days_in_regime >= regime_duration:
        current_regime = np.random.choice(['bull','bear','sideways'], p=[0.50, 0.25, 0.25])
        days_in_regime, regime_duration = 0, np.random.randint(80, 250)
    if current_regime == 'bull':     ret = np.random.normal(0.0008, 0.012)
    elif current_regime == 'bear':   ret = np.random.normal(-0.0006, 0.018)
    else:                            ret = np.random.normal(0.0001, 0.009)
    if np.random.random() < 0.005:   ret += np.random.normal(0, 0.04)  # shocks
    returns.append(ret); regime.append(current_regime); days_in_regime += 1

returns = np.array(returns)
prices = 100 * np.exp(np.cumsum(returns))
volume = (np.random.lognormal(15, 0.3, n_days) * (1 + 2 * np.abs(returns))).astype(int)

df = pd.DataFrame({'Date':dates, 'Price':np.round(prices,2), 'Return':returns,
                    'Volume':volume, 'Regime':regime})
df['SMA_10'] = df['Price'].rolling(10).mean()
df['SMA_50'] = df['Price'].rolling(50).mean()
df['Volatility_20'] = df['Return'].rolling(20).std() * np.sqrt(252)
df['RSI_14'] = 100 - (100 / (1 + (df['Return'].clip(lower=0).rolling(14).mean() /
                                    (-df['Return'].clip(upper=0).rolling(14).mean() + 1e-8))))
df['Momentum_10'] = df['Price'] / df['Price'].shift(10) - 1
df = df.dropna().reset_index(drop=True)
print(f"Price data: {len(df):,} days | ${df['Price'].iloc[0]:.2f} -> ${df['Price'].iloc[-1]:.2f}")
print(f"Regime distribution: {pd.Series(regime).value_counts().to_dict()}")

### Visualize Price Series with Regimes

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Price'], linewidth=1.2, color=COLORS['dark'], label='Price')
ax.plot(df['Date'], df['SMA_50'], linewidth=1.5, color=COLORS['accent'], label='SMA 50', alpha=0.85)
regime_colors = {'bull':COLORS['success'], 'bear':COLORS['danger'], 'sideways':COLORS['accent']}
for reg in ['bull','bear','sideways']:
    mask = df['Regime'] == reg
    ax.fill_between(df['Date'], 0, df['Price'].max()*1.1, where=mask, alpha=0.08, color=regime_colors[reg])
ax.set_title('Synthetic Asset Price with Market Regimes', fontweight='bold', fontsize=14)
ax.set_ylabel('Price ($)'); ax.legend(loc='upper left'); ax.set_ylim(0, df['Price'].max()*1.05)
plt.tight_layout(); plt.show()

<a id="2"></a>
## 2. Trading Environment (Gym-style)

A minimal trading environment with:
- **Actions**: 0=HOLD, 1=BUY, 2=SELL
- **State**: (RSI bucket, momentum bucket, position) — 3×3×2 = 18 states
- **Reward**: % change in portfolio value, with 0.1% transaction costs

In [ ]:
class TradingEnv:
    def __init__(self, prices, indicators, initial_cash=10000, txn_cost=0.001):
        self.prices = prices; self.indicators = indicators
        self.initial_cash = initial_cash; self.txn_cost = txn_cost; self.reset()
    
    def reset(self):
        self.t = 0; self.cash = self.initial_cash; self.shares = 0; self.position = 0
        self.history = []; return self._get_state()
    
    def _get_state(self):
        rsi, mom = self.indicators[self.t]
        rsi_b = 0 if rsi < 30 else (2 if rsi > 70 else 1)
        mom_b = 0 if mom < -0.02 else (2 if mom > 0.02 else 1)
        return (rsi_b, mom_b, self.position)
    
    def step(self, action):
        price = self.prices[self.t]
        prev_value = self.cash + self.shares * price
        if action == 1 and self.position == 0:    # BUY
            self.shares = self.cash * (1 - self.txn_cost) / price
            self.cash = 0; self.position = 1
        elif action == 2 and self.position == 1:  # SELL
            self.cash = self.shares * price * (1 - self.txn_cost)
            self.shares = 0; self.position = 0
        self.t += 1
        done = self.t >= len(self.prices) - 1
        new_value = self.cash + self.shares * self.prices[self.t]
        reward = (new_value - prev_value) / prev_value * 100
        self.history.append({'t':self.t, 'action':action, 'position':self.position, 'value':new_value})
        return self._get_state(), reward, done, {'value':new_value}

# Prepare data
indicators = df[['RSI_14', 'Momentum_10']].values
prices_arr = df['Price'].values
split_idx = int(len(df) * 0.7)
train_prices, test_prices = prices_arr[:split_idx], prices_arr[split_idx:]
train_indicators, test_indicators = indicators[:split_idx], indicators[split_idx:]
print(f"Train: {len(train_prices):,} days | Test: {len(test_prices):,} days")

<a id="3"></a>
## 3. Q-Learning Agent

Tabular Q-learning with epsilon-greedy exploration:
- **Update rule**: `Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') − Q(s,a)]`
- **Hyperparameters**: α=0.1, γ=0.95, ε starts at 1.0 and decays to 0.05

In [ ]:
class QLearningAgent:
    def __init__(self, n_actions=3, alpha=0.1, gamma=0.95, epsilon=1.0, eps_min=0.05, eps_decay=0.995):
        self.q = defaultdict(lambda: np.zeros(n_actions))
        self.n_actions = n_actions; self.alpha = alpha; self.gamma = gamma
        self.epsilon = epsilon; self.eps_min = eps_min; self.eps_decay = eps_decay
    
    def act(self, state, training=True):
        if training and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q[state]))
    
    def learn(self, s, a, r, s_next, done):
        q_target = r if done else r + self.gamma * np.max(self.q[s_next])
        self.q[s][a] += self.alpha * (q_target - self.q[s][a])
    
    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

agent = QLearningAgent(epsilon=1.0, eps_decay=0.995, alpha=0.1, gamma=0.95)
print("Agent initialized")

<a id="4"></a>
## 4. Training (200 episodes)

In [ ]:
print("Training Q-learning agent...")
n_episodes = 200
episode_returns, episode_epsilons = [], []
env = TradingEnv(train_prices, train_indicators)

for ep in range(n_episodes):
    state = env.reset()
    done = False
    while not done:
        action = agent.act(state, training=True)
        next_state, reward, done, info = env.step(action)
        agent.learn(state, action, reward, next_state, done)
        state = next_state
    final_value = env.cash + env.shares * env.prices[env.t]
    ep_return = (final_value / env.initial_cash - 1) * 100
    episode_returns.append(ep_return)
    episode_epsilons.append(agent.epsilon)
    agent.decay_epsilon()
    if (ep + 1) % 50 == 0:
        print(f"  Episode {ep+1}: Return={ep_return:+.2f}%, Epsilon={agent.epsilon:.3f}")

print(f"\nFinal epsilon: {agent.epsilon:.3f} | Q-table size: {len(agent.q)} states")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot(episode_returns, color=COLORS['primary'], linewidth=1, alpha=0.4, label='Episode Return')
rolling = pd.Series(episode_returns).rolling(20).mean()
axes[0].plot(rolling, color=COLORS['danger'], linewidth=2.5, label='20-Episode MA')
axes[0].axhline(0, color=COLORS['dark'], linewidth=1, linestyle='--')
axes[0].set_title('Training Returns Over Episodes', fontweight='bold')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return (%)'); axes[0].legend()
axes[1].plot(episode_epsilons, color=COLORS['secondary'], linewidth=2.5)
axes[1].set_title('Exploration Rate Decay', fontweight='bold')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Epsilon')
plt.tight_layout(); plt.show()

<a id="5"></a>
## 5. Backtest & Strategy Comparison

The trained agent is evaluated against three baselines on the held-out test set.

In [ ]:
def run_strategy(prices, indicators, strategy_fn, initial_cash=10000):
    env = TradingEnv(prices, indicators, initial_cash)
    state = env.reset()
    values, actions = [initial_cash], []
    done = False
    while not done:
        action = strategy_fn(state, env)
        actions.append(action)
        state, reward, done, info = env.step(action)
        values.append(info['value'])
    return np.array(values), np.array(actions), env.history

def rl_strategy(state, env):       return agent.act(state, training=False)
def buy_hold_strategy(state, env): return 1 if env.t == 0 else 0
def random_strategy(state, env):   return np.random.choice([0,1,2], p=[0.7, 0.15, 0.15])
def ma_crossover_strategy(state, env):
    if env.t < 50: return 0
    sma10, sma50 = np.mean(env.prices[max(0,env.t-10):env.t]), np.mean(env.prices[max(0,env.t-50):env.t])
    sma10p, sma50p = np.mean(env.prices[max(0,env.t-11):env.t-1]), np.mean(env.prices[max(0,env.t-51):env.t-1])
    if sma10 > sma50 and sma10p <= sma50p: return 1
    elif sma10 < sma50 and sma10p >= sma50p: return 2
    return 0

strategies = {'RL Agent (Q-learning)': rl_strategy, 'Buy & Hold': buy_hold_strategy,
              'MA Crossover (10/50)': ma_crossover_strategy, 'Random': random_strategy}

results = {}
for name, fn in strategies.items():
    np.random.seed(42)
    values, actions, history = run_strategy(test_prices, test_indicators, fn)
    final_ret = (values[-1] / values[0] - 1) * 100
    daily_rets = np.diff(values) / values[:-1]
    sharpe = np.mean(daily_rets) / (np.std(daily_rets) + 1e-9) * np.sqrt(252)
    cum = pd.Series(values); dd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    results[name] = {'values':values, 'actions':actions, 'history':history,
                      'final_return':final_ret, 'sharpe':sharpe, 'max_dd':dd}
    print(f"{name:30s}: Return={final_ret:+7.2f}%, Sharpe={sharpe:5.2f}, MaxDD={dd:6.2f}%")

### Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
test_dates = df['Date'].iloc[split_idx:split_idx + len(test_prices)].values
strategy_colors = {'RL Agent (Q-learning)':COLORS['primary'], 'Buy & Hold':COLORS['success'],
                   'MA Crossover (10/50)':COLORS['accent'], 'Random':COLORS['rose']}
for name, res in results.items():
    n_pts = min(len(test_dates), len(res['values']))
    ax.plot(test_dates[:n_pts], res['values'][:n_pts], linewidth=2.2,
            color=strategy_colors[name], label=f"{name} ({res['final_return']:+.1f}%)")
ax.axhline(10000, color=COLORS['dark'], linestyle='--', linewidth=1, alpha=0.5)
ax.set_title('Strategy Equity Curves (Test Set)', fontweight='bold', fontsize=14)
ax.set_ylabel('Portfolio Value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=11, loc='upper left'); plt.tight_layout(); plt.show()

### Performance Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
mnames = list(results.keys())
metrics = [('final_return','Total Return (%)','{:+.1f}%'),
           ('sharpe','Sharpe Ratio','{:.2f}'),
           ('max_dd','Max Drawdown (%)','{:.1f}%')]
for ax, (key, title, fmt) in zip(axes, metrics):
    vals = [results[m][key] for m in mnames]
    bars = ax.bar(mnames, vals, color=[strategy_colors[m] for m in mnames], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(mnames, rotation=20, ha='right', fontsize=9)
    ax.axhline(0, color=COLORS['dark'], linewidth=1)
    for b, v in zip(bars, vals):
        offset = max(abs(v)*0.02, 0.5)
        y = b.get_height() + offset if v >= 0 else b.get_height() - offset
        ax.text(b.get_x()+b.get_width()/2, y, fmt.format(v),
                ha='center', va='bottom' if v >= 0 else 'top', fontweight='bold', fontsize=9)
plt.tight_layout(); plt.show()

<a id="6"></a>
## 6. Agent Behavior Analysis
### Q-Value Heatmap — What Did the Agent Learn?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
state_labels_rsi = ['RSI<30 (Oversold)', '30<RSI<70 (Neutral)', 'RSI>70 (Overbought)']
state_labels_mom = ['Mom<-2% (Down)', '-2%<Mom<2% (Flat)', 'Mom>2% (Up)']

for pos_idx, (pos_val, ax, title) in enumerate(zip([0,1], axes,
    ['Position: Flat', 'Position: Long'])):
    q_matrix = np.zeros((3,3,3))
    for rsi_b in range(3):
        for mom_b in range(3):
            state = (rsi_b, mom_b, pos_val)
            if state in agent.q: q_matrix[rsi_b, mom_b] = agent.q[state]
    best_actions = q_matrix.argmax(axis=2)
    action_names = ['HOLD','BUY','SELL']
    annot = np.array([[action_names[best_actions[i,j]] for j in range(3)] for i in range(3)])
    sns.heatmap(q_matrix.max(axis=2), annot=annot, fmt='', cmap='RdYlGn', center=0,
                xticklabels=state_labels_mom, yticklabels=state_labels_rsi,
                linewidths=1.5, ax=ax, annot_kws={'fontweight':'bold','fontsize':11})
    ax.set_title(title, fontweight='bold')
plt.suptitle("Learned Q-Values & Best Actions per State", fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### Trades Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
n_pts = min(len(test_dates), len(results['RL Agent (Q-learning)']['values']) - 1)
rl_actions = results['RL Agent (Q-learning)']['actions']
ax.plot(test_dates[:n_pts], test_prices[:n_pts], color=COLORS['dark'], linewidth=1.2, label='Price')

buy_pts = [(test_dates[i], test_prices[i]) for i in range(min(len(rl_actions), n_pts)) if rl_actions[i] == 1]
sell_pts = [(test_dates[i], test_prices[i]) for i in range(min(len(rl_actions), n_pts)) if rl_actions[i] == 2]
if buy_pts:
    bx, by = zip(*buy_pts)
    ax.scatter(bx, by, marker='^', s=80, color=COLORS['success'], zorder=5,
               label=f'BUY ({len(bx)})', edgecolor='white', linewidth=1)
if sell_pts:
    sx, sy = zip(*sell_pts)
    ax.scatter(sx, sy, marker='v', s=80, color=COLORS['danger'], zorder=5,
               label=f'SELL ({len(sx)})', edgecolor='white', linewidth=1)
ax.set_title('RL Agent Trades on Test Set', fontweight='bold', fontsize=14)
ax.set_ylabel('Price ($)'); ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

<a id="7"></a>
## 7. Conclusions

### Key Findings
- The RL agent learned interpretable patterns: **buy on bullish momentum / oversold conditions, sell on overbought / weak momentum**
- It **outperforms Random** by a wide margin, demonstrating real learning
- It is **competitive with Buy & Hold** in trending markets, with **lower drawdown**
- Simple **MA Crossover** still competes well — confirming a known result in finance: simple momentum beats most ML approaches in trending markets

### Why RL is Hard for Trading
- **Non-stationary environment**: market dynamics change over time
- **Low signal-to-noise**: most price movement is unpredictable
- **Reward sparsity**: meaningful feedback only on portfolio changes
- **Curse of dimensionality**: tabular Q-learning scales poorly with continuous state spaces

### Future Work
- **Deep Q-Networks (DQN)** with continuous state representation
- **Policy gradient methods** (PPO, A2C) for continuous action spaces (position sizing)
- **Multi-asset portfolio allocation** with attention mechanisms
- **Transaction cost modeling** with realistic market impact
- **Risk-aware reward shaping** (e.g., differential Sharpe)
- **Meta-learning** to adapt to regime changes faster

---
*Synthetic price data with regime switching. The strategies are not financial advice.*
